# FlyRank Search Intelligence Capstone: Search Ranking Prediction Pipeline

## Abstract
This standalone Jupyter Notebook encapsulates the complete, end-to-end Machine Learning pipeline for the **FlyRank Search Intelligence Capstone Project**. The objective is to predict web page search ranking positions (`avg_position_90d`) from raw search engine logs hosted in a remote Hugging Face Parquet warehouse. The notebook executes three primary stages:
1. **Data Ingestion & Aggregation**: Remote DuckDB warehouse query joining `fact_content_query_90d` and `dim_content`.
2. **Feature Engineering**: Normalized log transformation, baseline CTR computation, interaction density calculation, and dataset export (`X_train.csv`, `y_train.csv`).
3. **Model Sweeps & Leaderboard**: Evaluation of multiple regressors (`RandomForestRegressor`, `GradientBoostingRegressor`, `LinearRegression`, `Ridge`), leaderboard ranking, and champion model serialization (`best_search_ranking_model.pkl`).

## Section 1: Data Ingestion & Aggregation

In this section, we set up authentication via Hugging Face token (if present in environment or `token.txt`), initialize an in-memory DuckDB connection with `httpfs`, and execute a SQL query on remote Parquet files to extract daily search facts joined with content dimensions.

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np

# 1. Look for token.txt or environment variable HF_TOKEN
token = os.environ.get("HF_TOKEN")
if not token and os.path.exists("token.txt"):
    with open("token.txt", "r") as f:
        token = f.read().strip()

# 2. Initialize DuckDB connection
print("Initializing DuckDB connection...")
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# 3. Configure Hugging Face token if present
if token:
    print("Hugging Face token found. Configuring authentication secret...")
    escaped_token = token.replace("'", "''")
    con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{escaped_token}');")
else:
    print("No Hugging Face token found in HF_TOKEN or token.txt.")
    print("Attempting to query without authentication...")

# 4. Pull sample records from dataset using glob pattern
dataset_pattern = "hf://datasets/FlyRank/internship-warehouse/**/*.parquet"
print(f"Querying data sample from: {dataset_pattern}")

ingest_query = f"""
    WITH all_data AS (
        SELECT *, filename FROM read_parquet('{dataset_pattern}', filename=True, union_by_name=True)
    )
    SELECT 
        q.query_hash_id AS query, 
        c.url_hash_id AS url, 
        q.clicks_90d AS clicks, 
        q.impressions_90d AS impressions
    FROM (SELECT * FROM all_data WHERE filename LIKE '%fact_content_query_90d%') q
    JOIN (SELECT * FROM all_data WHERE filename LIKE '%dim_content%') c
      ON q.content_hash_id = c.content_hash_id
    LIMIT 1000
"""

df_ingest = con.execute(ingest_query).df()
print(f"Successfully pulled {len(df_ingest)} rows for ingestion test.")

# 5. Aggregate by query and URL and compute baseline CTR
df_ingest['clicks'] = pd.to_numeric(df_ingest['clicks'], errors='coerce').fillna(0)
df_ingest['impressions'] = pd.to_numeric(df_ingest['impressions'], errors='coerce').fillna(0)

aggregated = df_ingest.groupby(['query', 'url']).agg({
    'clicks': 'sum',
    'impressions': 'sum'
}).reset_index()

aggregated["baseline_ctr"] = aggregated['clicks'] / aggregated['impressions'].replace(0, 1)
print("\nData Ingestion & Aggregation Sample:")
print(aggregated.head())


## Section 2: Feature Engineering

In this section, we extract a dataset of 10,000 records including target `avg_position_90d` and engineer three primary numerical features:
- `log_impressions`: $\log(impressions + 1)$ to normalize extreme search volume skewness.
- `baseline_ctr`: $clicks / impressions$ capturing baseline click rates.
- `interaction_density`: $clicks \times log\_impressions$ measuring joint search interaction magnitude.

The resulting matrices are exported to `X_train.csv` and `y_train.csv`.

In [ ]:
# Query expanded 10,000 rows sample for feature engineering
feature_query = f"""
    WITH all_data AS (
        SELECT *, filename FROM read_parquet('{dataset_pattern}', filename=True, union_by_name=True)
    )
    SELECT 
        q.query_hash_id AS query, 
        c.url_hash_id AS url, 
        q.clicks_90d AS clicks, 
        q.impressions_90d AS impressions,
        q.avg_position_90d AS average_position
    FROM (SELECT * FROM all_data WHERE filename LIKE '%fact_content_query_90d%') q
    JOIN (SELECT * FROM all_data WHERE filename LIKE '%dim_content%') c
      ON q.content_hash_id = c.content_hash_id
    LIMIT 10000
"""

print(f"Querying expanded sample of 10,000 rows for feature matrix creation...")
df_feat = con.execute(feature_query).df()
print(f"Successfully pulled {len(df_feat)} rows.")

# Convert numeric columns safely
df_feat['clicks'] = pd.to_numeric(df_feat['clicks'], errors='coerce').fillna(0)
df_feat['impressions'] = pd.to_numeric(df_feat['impressions'], errors='coerce').fillna(0)
df_feat['average_position'] = pd.to_numeric(df_feat['average_position'], errors='coerce').fillna(0)

# Calculate features
df_feat['log_impressions'] = np.log1p(df_feat['impressions'])
df_feat['baseline_ctr'] = (df_feat['clicks'] / df_feat['impressions'].replace(0, 1)).fillna(0.0)
df_feat['interaction_density'] = df_feat['clicks'] * df_feat['log_impressions']

# Prepare X and y
feature_cols = ['log_impressions', 'baseline_ctr', 'interaction_density']
X = df_feat[feature_cols]
y = df_feat['average_position']

# Save to local workspace
X.to_csv("X_train.csv", index=False)
y.to_csv("y_train.csv", index=False)

print(f"Feature matrix X shape: {X.shape}, Target array y shape: {y.shape}")
print("\nFirst 5 rows of feature matrix X:")
print(X.head())


## Section 3: Model Sweeps & Leaderboard

In this section, we split the feature matrix into an **80% training / 20% validation split** (with `random_state=42`) and execute a model sweep across four algorithm architectures:
- **RandomForestRegressor**
- **GradientBoostingRegressor**
- **LinearRegression**
- **Ridge**

We evaluate models using Root Mean Squared Error (RMSE) and $R^2$ Score, construct a formatted leaderboard, identify the champion model, and persist it to `best_search_ranking_model.pkl` via `joblib`.

In [ ]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge

# 1. Load feature data
print("Loading X_train.csv and y_train.csv...")
X = pd.read_csv("X_train.csv")
y = pd.read_csv("y_train.csv")
y = y.iloc[:, 0] if len(y.shape) > 1 else y

# 2. Train / Validation split (80/20)
print("Splitting data into 80% training and 20% validation sets...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Define candidate models
models = {
    "RandomForestRegressor": RandomForestRegressor(random_state=42),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42),
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge()
}

# 4. Iterate and evaluate models
results = []
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    
    print(f"Evaluating {name}...")
    y_pred = model.predict(X_val)
    
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)
    
    results.append({
        "Model": name,
        "RMSE": rmse,
        "R2": r2
    })
    trained_models[name] = model

# 5. Display Leaderboard
df_results = pd.DataFrame(results).sort_values(by="RMSE", ascending=True).reset_index(drop=True)

print("\n================== MODEL LEADERBOARD ==================")
print(df_results.to_string(index=False))
print("=======================================================")

# 6. Programmatically identify and serialize winning model
best_model_info = df_results.iloc[0]
best_model_name = best_model_info["Model"]
best_model_rmse = best_model_info["RMSE"]
best_model_r2 = best_model_info["R2"]

print(f"\nBest Model Identified: {best_model_name} (RMSE: {best_model_rmse:.4f}, R2: {best_model_r2:.4f})")

best_model = trained_models[best_model_name]
model_filename = "best_search_ranking_model.pkl"
print(f"Saving winning model to {model_filename}...")
joblib.dump(best_model, model_filename)
print("Model saved successfully!")
